<a href="https://colab.research.google.com/github/GVimalKaushik/Hate-Speech-Detection-Using-Deep-Learning/blob/main/Hate_Speech_Detection_using_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langdetect regex transformers torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 23.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=02214fec456bb9e3e53f45cfebdc5c6d5724e0da4da05ae887df7a953e06eccd
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
import re
import unicodedata
import regex as reg
class TextPreprocessingLayer:
    def __init__(self):
        self.slang_dict = {
            "u": "you",
            "ur": "your",
            "lol": "laughing out loud",
            "omg": "oh my god",
            "idk": "i do not know",
            "wtf": "what the fuck",
            "bc": "because",
            "btw": "by the way",
            "imo": "in my opinion",
            "r": "are",
            "n": "and"
        }
        self.script_patterns = {
            "Latin": reg.compile(r"\p{Latin}"),
            "Devanagari": reg.compile(r"\p{Devanagari}"),
            "Arabic": reg.compile(r"\p{Arabic}"),
            "Cyrillic": reg.compile(r"\p{Cyrillic}"),
            "Han": reg.compile(r"\p{Han}")
        }
    def identify_script(self, text):
        script_count = {}
        for script, pattern in self.script_patterns.items():
            script_count[script] = len(pattern.findall(text))
        detected_scripts = [s for s, count in script_count.items() if count > 0]
        if len(detected_scripts) > 1:
            return "Mixed"
        elif len(detected_scripts) == 1:
            return detected_scripts[0]
        else:
            return "Unknown"
    def normalize_text(self, text, script):
        text = unicodedata.normalize("NFKC", text)
        text = re.sub(r"http\S+|www\S+", "", text)
        text = re.sub(r"@\w+|#\w+", "", text)
        text = reg.sub(r"(.)\1{2,}", r"\1\1", text)
        if script in ["Latin", "Mixed"]:
            text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
            text = text.lower()
        else:
            text = reg.sub(r"[^\p{L}\p{M}\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text
    def remove_slang(self, text):
        words = text.split()
        expanded_words = []
        for word in words:
            expanded_words.append(self.slang_dict.get(word, word))
        return " ".join(expanded_words)
    def preprocess(self, raw_text):
        script = self.identify_script(raw_text)
        normalized_text = self.normalize_text(raw_text, script)
        clean_text = self.remove_slang(normalized_text)
        return clean_text

In [ ]:
from transformers import AutoTokenizer
import torch
class TokenizationLayer:
    def __init__(self, model_name="bert-base-multilingual-uncased", max_length=48):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.max_length = max_length
    def encode(self, text):
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
            return_attention_mask=True,
            return_token_type_ids=True
        )
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "token_type_ids": encoding["token_type_ids"]
        }

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
class MBERTEncoderLayer(nn.Module):
    def __init__(self, model_name="bert-base-multilingual-uncased"):
        super(MBERTEncoderLayer, self).__init__()

        self.bert = AutoModel.from_pretrained(model_name)
        # Freeze first 9 transformer layers for faster training
        for param in self.bert.encoder.layer[:9].parameters():
            param.requires_grad = False
    def forward(self, encoded_inputs):
        outputs = self.bert(
            input_ids=encoded_inputs["input_ids"],
            attention_mask=encoded_inputs["attention_mask"],
            token_type_ids=encoded_inputs["token_type_ids"]
        )
        last_hidden_state = outputs.last_hidden_state
        """
        print("\nLayer 3 Output (Last Hidden State Embeddings):")
        print("Last Hidden State shape:", last_hidden_state.shape)
        """
        return last_hidden_state

In [ ]:
import torch
import torch.nn as nn
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None
class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_=0.5):
        super().__init__()
        self.lambda_ = lambda_
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_)
class BiasDiscriminator(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=128, num_bias_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_bias_classes)
        )
    def forward(self, x):
        return self.net(x)
class AdversarialDebiasingLayer(nn.Module):
    def __init__(self, embedding_dim=768, lambda_=0.0):
        super().__init__()
        self.lambda_ = lambda_
        self.grl = GradientReversalLayer(self.lambda_)
        self.discriminator = BiasDiscriminator(embedding_dim)
    def set_lambda(self, lambda_):
        self.lambda_ = lambda_
        self.grl.lambda_ = lambda_
    def forward(self, last_hidden_state):
        sentence_embedding = last_hidden_state[:, 0, :]
        reversed_embedding = self.grl(sentence_embedding)
        bias_logits = self.discriminator(reversed_embedding)
        return last_hidden_state, bias_logits

In [ ]:
import torch
import torch.nn as nn
class TaskSpecificTransformerEncoder(nn.Module):
    def __init__(self, embedding_dim=768, num_heads=8, num_layers=2,
                 dim_feedforward=2048, dropout=0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
    def forward(self, token_embeddings, attention_mask=None):
        if attention_mask is not None:
            padding_mask = attention_mask == 0
        else:
            padding_mask = None
        task_refined_embeddings = self.transformer_encoder(
            token_embeddings,
            src_key_padding_mask=padding_mask
        )
        return task_refined_embeddings

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class ClassificationLayer(nn.Module):
    def __init__(self, embedding_dim=768, num_classes=2):
        super(ClassificationLayer, self).__init__()
        self.dense = nn.Linear(embedding_dim, num_classes)
    def forward(self, task_refined_embeddings):
        cls_embedding = task_refined_embeddings[:, 0, :]
        logits = self.dense(cls_embedding)
        probs = F.softmax(logits, dim=1)
        """
        print("\nLAYER 6 OUTPUT")
        print("CLS Embedding shape:", cls_embedding.shape)
        print("Logits shape:", logits.shape)
        print("Sample logits:", logits[0].detach().cpu())
        print("Softmax probabilities:", probs[0].detach().cpu())
        """
        return logits, probs

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes, smoothing=0.1, dim=-1):
        super(LabelSmoothingLoss, self).__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
        self.dim = dim
    def forward(self, pred, target):
        log_probs = pred.log_softmax(dim=self.dim)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (self.cls - 1))
            true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
        loss = torch.mean(torch.sum(-true_dist * log_probs, dim=self.dim))
        return loss

In [ ]:
import torch
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
class ModelEvaluation:
    def __init__(self, num_classes=2):
        self.num_classes = num_classes
    def evaluate(self, true_labels, probs):
        if isinstance(true_labels, torch.Tensor):
            true_labels = true_labels.detach().cpu().numpy()
        if isinstance(probs, torch.Tensor):
            probs = probs.detach().cpu().numpy()
        predicted_labels = np.argmax(probs, axis=1)
        cm = confusion_matrix(true_labels, predicted_labels)
        accuracy = accuracy_score(true_labels, predicted_labels)
        precision = precision_score(true_labels, predicted_labels, average='macro', zero_division=0)
        recall = recall_score(true_labels, predicted_labels, average='macro', zero_division=0)
        f1 = f1_score(true_labels, predicted_labels, average='macro', zero_division=0)
        print("\nEvaluation Metrics")
        print("------------------")
        print("Confusion Matrix:\n", cm)
        print("Accuracy:", round(accuracy, 4))
        print("Precision (Macro):", round(precision, 4))
        print("Recall (Macro):", round(recall, 4))
        print("F1 Score (Macro):", round(f1, 4))
        return {
            "confusion_matrix": cm,
            "accuracy": accuracy,
            "precision_macro": precision,
            "recall_macro": recall,
            "f1_macro": f1
        }

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
from google.colab import files
from torch.utils.data import Dataset, DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()
epochs = 6
batch_size = 32
layer1 = TextPreprocessingLayer()
layer2 = TokenizationLayer(max_length=48)
layer3 = MBERTEncoderLayer().to(device)
layer4 = AdversarialDebiasingLayer(embedding_dim=768, lambda_=0.0).to(device)
layer5 = TaskSpecificTransformerEncoder().to(device)
layer6 = ClassificationLayer(768, 2).to(device)
layer7 = LabelSmoothingLoss(classes=2)
task_criterion = layer7
bias_criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.AdamW(
    list(layer3.parameters()) +
    list(layer4.parameters()) +
    list(layer5.parameters()) +
    list(layer6.parameters()),
    lr=2e-5
)
scaler = torch.amp.GradScaler('cuda') if use_amp else None
class HateSpeechDatasetDebias(Dataset):
    def __init__(self, texts, labels, bias_labels):
        self.texts = texts
        self.labels = labels
        self.bias_labels = bias_labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        clean_text = layer1.preprocess(str(self.texts[idx]))
        encoded = layer2.encode(clean_text)
        token_type_ids = encoded.get(
            "token_type_ids",
            torch.zeros_like(encoded["input_ids"], dtype=torch.long)
        )
        return (
            encoded["input_ids"].squeeze(0),
            encoded["attention_mask"].squeeze(0),
            token_type_ids.squeeze(0),
            torch.tensor(int(self.labels[idx])).long(),
            torch.tensor(int(self.bias_labels[idx])).long()
        )
def evaluate_model(loader):
            all_preds = []
            all_labels = []
            bias_preds_all = []
            bias_labels_all = []

            with torch.no_grad():
                for i_ids, a_mask, t_type, l_t, bias_t in loader:
                    i_ids = i_ids.to(device)
                    a_mask = a_mask.to(device)
                    t_type = t_type.to(device)

                    lh = layer3({
                        "input_ids": i_ids,
                        "attention_mask": a_mask,
                        "token_type_ids": t_type
                    })

                    db, bias_logits = layer4(lh)
                    te = layer5(db, a_mask)
                    _, probs = layer6(te)

                    preds = torch.argmax(probs, dim=1)
                    bias_preds = torch.argmax(bias_logits, dim=1)

                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(l_t.cpu().numpy())
                    bias_preds_all.extend(bias_preds.cpu().numpy())
                    bias_labels_all.extend(bias_t.cpu().numpy())

            return {
                "accuracy": accuracy_score(all_labels, all_preds),
                "precision": precision_score(all_labels, all_preds, zero_division=0),
                "recall": recall_score(all_labels, all_preds, zero_division=0),
                "f1": f1_score(all_labels, all_preds, zero_division=0),
                "bias_acc": accuracy_score(bias_labels_all, bias_preds_all),
                "cm": confusion_matrix(all_labels, all_preds)
            }
mode = input("Enter mode (train/test): ").strip().lower()
if mode == "train":
    print("\nUpload HateSpeechDataset.csv")
    uploaded = files.upload()
    file_path = list(uploaded.keys())[0]
    data = pd.read_csv(file_path)
    data.columns = data.columns.str.strip()
    data = data.dropna(subset=["Content", "Label", "bias_label"])
    data["Label"] = data["Label"].astype(int)
    data["bias_label"] = data["bias_label"].astype(int)
    texts = data["Content"].astype(str).tolist()
    labels = data["Label"].values
    bias_labels = data["bias_label"].values
    train_texts, test_texts, train_labels, test_labels, train_bias, test_bias = train_test_split(
        texts, labels, bias_labels, test_size=0.2, random_state=42
    )
    train_loader = DataLoader(
        HateSpeechDatasetDebias(train_texts, train_labels, train_bias),
        batch_size=batch_size,
        shuffle=True
    )
    test_loader = DataLoader(
        HateSpeechDatasetDebias(test_texts, test_labels, test_bias),
        batch_size=batch_size
    )
    for epoch in range(epochs):
        p = epoch / (epochs - 1)
        lambda_val = 2 / (1 + np.exp(-10 * p)) - 1  # smooth curve from 0 → 1
        layer4.set_lambda(lambda_val)
        print(f"Epoch {epoch+1} | Lambda: {lambda_val:.4f}")
        layer3.train(); layer4.train(); layer5.train(); layer6.train()
        total_loss = 0
        for input_ids, att_mask, tok_type, labels_t, bias_t in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            input_ids = input_ids.to(device)
            att_mask = att_mask.to(device)
            tok_type = tok_type.to(device)
            labels_t = labels_t.to(device)
            bias_t = bias_t.to(device)
            optimizer.zero_grad()
            if use_amp:
                with torch.amp.autocast('cuda'):
                    lh = layer3({
                        "input_ids": input_ids,
                        "attention_mask": att_mask,
                        "token_type_ids": tok_type
                    })
                    db, bias_logits = layer4(lh)
                    te = layer5(db, att_mask)
                    logits, probs = layer6(te)
                adv_loss = bias_criterion(bias_logits.float(), bias_t)
                task_loss = task_criterion(logits.float(), labels_t)
                alpha = 0.5   # you can tune this (0.1–1.0)
                loss = task_loss + alpha * adv_loss
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                lh = layer3({
                    "input_ids": input_ids,
                    "attention_mask": att_mask,
                    "token_type_ids": tok_type
                })
                db, bias_logits = layer4(lh)
                te = layer5(db, att_mask)
                logits, probs = layer6(te)
                adv_loss = bias_criterion(bias_logits, bias_t)
                task_loss = task_criterion(logits, labels_t)
                alpha = 0.5   # you can tune this (0.1–1.0)
                loss = task_loss + alpha * adv_loss
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
        layer3.eval(); layer4.eval(); layer5.eval(); layer6.eval()
        train_metrics = evaluate_model(train_loader)
        test_metrics = evaluate_model(test_loader)
        print(f"\n--- Epoch {epoch+1} Results ---")
        print("\n🔹 Train Metrics:")
        print("Accuracy:", train_metrics["accuracy"])
        print("Precision:", train_metrics["precision"])
        print("Recall:", train_metrics["recall"])
        print("F1 Score:", train_metrics["f1"])
        print("Bias Accuracy:", train_metrics["bias_acc"])
        print("\n🔹 Test Metrics:")
        print("Accuracy:", test_metrics["accuracy"])
        print("Precision:", test_metrics["precision"])
        print("Recall:", test_metrics["recall"])
        print("F1 Score:", test_metrics["f1"])
        print("Bias Accuracy:", test_metrics["bias_acc"])
    torch.save({
        "layer3": layer3.state_dict(),
        "layer4": layer4.state_dict(),
        "layer5": layer5.state_dict(),
        "layer6": layer6.state_dict()
    }, "hate_speech_model.pt")
elif mode == "test":
    checkpoint = torch.load("hate_speech_model.pt", map_location=device)
    layer3.load_state_dict(checkpoint["layer3"])
    layer4.load_state_dict(checkpoint["layer4"])
    layer5.load_state_dict(checkpoint["layer5"])
    layer6.load_state_dict(checkpoint["layer6"])
    layer3.eval(); layer4.eval(); layer5.eval(); layer6.eval()
    layer4.set_lambda(0.0)
    while True:
        print("\n1 → Enter text")
        print("2 → Upload file")
        print("exit → Quit")
        choice = input("Choice: ").strip().lower()
        if choice == "exit":
            break
        elif choice == "1":
            text = input("Enter text: ")
            clean = layer1.preprocess(text)
            enc = layer2.encode(clean)
            for k in enc:
                enc[k] = enc[k].to(device)
            with torch.no_grad():
                lh = layer3(enc)
                db, bias_logits = layer4(lh)
                te = layer5(db, enc["attention_mask"])
                _, probs = layer6(te)
                pred = torch.argmax(probs, dim=1).item()
            print(f"Prediction: {'Hate' if pred==1 else 'No Hate'} ({probs[0][1].item()*100:.2f}%)")
        elif choice == "2":
            uploaded = files.upload()
            file_path = list(uploaded.keys())[0]

            if file_path.endswith(".csv"):
                df = pd.read_csv(file_path)
                df.columns = df.columns.str.strip()

                if "Content" not in df.columns or "Label" not in df.columns:
                    raise ValueError("CSV must contain 'Content' and 'Label' columns")

                texts = df["Content"].astype(str).tolist()
                true_labels = df["Label"].astype(int).tolist()

        # Optional bias labels
                has_bias = "bias_label" in df.columns
                if has_bias:
                    true_bias = df["bias_label"].astype(int).tolist()

                all_preds = []
                all_probs = []
                bias_preds_all = []

                print("\nProcessing CSV...\n")

                for i, text in enumerate(tqdm(texts)):
                    clean = layer1.preprocess(text)
                    enc = layer2.encode(clean)

                    for k in enc:
                        enc[k] = enc[k].to(device)

                    with torch.no_grad():
                        lh = layer3(enc)
                        db, bias_logits = layer4(lh)
                        te = layer5(db, enc["attention_mask"])
                        _, probs = layer6(te)

                        pred = torch.argmax(probs, dim=1).item()
                        confidence = probs[0][1].item() * 100

                        bias_pred = torch.argmax(bias_logits, dim=1).item()

                    all_preds.append(pred)
                    all_probs.append(confidence)
                    bias_preds_all.append(bias_pred)

        # 🔥 METRICS
                print("\n--- Overall Metrics ---")
                print("Accuracy:", accuracy_score(true_labels, all_preds))
                print("Precision:", precision_score(true_labels, all_preds, zero_division=0))
                print("Recall:", recall_score(true_labels, all_preds, zero_division=0))
                print("F1 Score:", f1_score(true_labels, all_preds, zero_division=0))
                print("Confusion Matrix:\n", confusion_matrix(true_labels, all_preds))

                if has_bias:
                    print("Bias Accuracy:", accuracy_score(true_bias, bias_preds_all))

            elif file_path.endswith(".txt"):
        # keep your existing txt logic
                with open(file_path, "r", encoding="utf-8") as f:
                    texts = [line.strip() for line in f if line.strip() != ""]

                print("\nPredictions:\n")

                for text in tqdm(texts):
                    clean = layer1.preprocess(text)
                    enc = layer2.encode(clean)

                    for k in enc:
                        enc[k] = enc[k].to(device)

                    with torch.no_grad():
                        lh = layer3(enc)
                        db, bias_logits = layer4(lh)
                        te = layer5(db, enc["attention_mask"])
                        _, probs = layer6(te)

                        pred = torch.argmax(probs, dim=1).item()
                        confidence = probs[0][1].item() * 100

                    print(f"{text} | {pred} | {confidence:.2f}%")

            else:
                raise ValueError("Only .csv or .txt files are allowed")
        else:
            print("Invalid choice")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Enter mode (train/test): test

1 → Enter text
2 → Upload file
exit → Quit
Choice: exit


In [ ]:
# 🔷 Interactive Integration Testing Script

import torch

def integration_test(text):
    print("\n" + "="*60)
    print("🔷 INTEGRATION TEST STARTED")
    print("="*60)

    print("\n🔹 Step 1: Raw Input")
    print(text)

    # Step 2: Preprocessing
    clean = layer1.preprocess(text)
    print("\n🔹 Step 2: After Preprocessing")
    print(clean)

    # Step 3: Tokenization + Encoding
    enc = layer2.encode(clean)
    print("\n🔹 Step 3: Tokenization & Encoding")
    print("Input IDs Shape:", enc["input_ids"].shape)
    print("Attention Mask Shape:", enc["attention_mask"].shape)

    # Move to device
    for k in enc:
        enc[k] = enc[k].to(device)

    with torch.no_grad():

        # Step 4: mBERT Encoder
        lh = layer3(enc)
        print("\n🔹 Step 4: mBERT Output")
        print("Shape:", lh.shape)

        # Step 5: Adversarial Debiasing
        db, bias_logits = layer4(lh)
        print("\n🔹 Step 5: Debiasing Layer Output")
        print("Debiased Shape:", db.shape)
        print("Bias Logits Shape:", bias_logits.shape)

        # Step 6: Transformer Encoder
        te = layer5(db, enc["attention_mask"])
        print("\n🔹 Step 6: Transformer Encoder Output")
        print("Shape:", te.shape)

        # Step 7: Classification Layer
        logits, probs = layer6(te)
        print("\n🔹 Step 7: Classification Output")
        print("Logits:", logits)
        print("Probabilities:", probs)

        # Final Prediction
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][1].item() * 100

    print("\n🔹 Final Prediction")
    print(f"Class: {'Hate Speech' if pred==1 else 'Non-Hate'}")
    print(f"Confidence: {confidence:.2f}%")

    print("\n🔷 INTEGRATION TEST COMPLETED")
    print("="*60)


# 🔷 LOOP FOR MANUAL INPUT
while True:
    print("\nEnter a sentence (or type 'exit' to stop):")
    user_input = input(">>> ")

    if user_input.lower() == "exit":
        print("🔴 Exiting Integration Testing...")
        break

    integration_test(user_input)


Enter a sentence (or type 'exit' to stop):
>>> Go back to where u came from! U r traitors

🔷 INTEGRATION TEST STARTED

🔹 Step 1: Raw Input
Go back to where u came from! U r traitors

🔹 Step 2: After Preprocessing
go back to where you came from you are traitors

🔹 Step 3: Tokenization & Encoding
Input IDs Shape: torch.Size([1, 48])
Attention Mask Shape: torch.Size([1, 48])

🔹 Step 4: mBERT Output
Shape: torch.Size([1, 48, 768])

🔹 Step 5: Debiasing Layer Output
Debiased Shape: torch.Size([1, 48, 768])
Bias Logits Shape: torch.Size([1, 3])

🔹 Step 6: Transformer Encoder Output
Shape: torch.Size([1, 48, 768])

🔹 Step 7: Classification Output
Logits: tensor([[-1.4499,  0.4041]], device='cuda:0')
Probabilities: tensor([[0.1354, 0.8646]], device='cuda:0')

🔹 Final Prediction
Class: Hate Speech
Confidence: 86.46%

🔷 INTEGRATION TEST COMPLETED

Enter a sentence (or type 'exit' to stop):
>>> exit
🔴 Exiting Integration Testing...
